In [0]:
import pandas as pd

In [0]:
cc_calls_df = pd.read_csv("../../data/raw/cc_calls.csv")
cc_calls_df.head()

In [0]:
print(f"Shape: {cc_calls_df.shape}\n")
cc_calls_df.info()

In [0]:
# Check unique values for all columns
for col in cc_calls_df.columns:
    n_unique = cc_calls_df[col].nunique()
    sample_vals = cc_calls_df[col].dropna().unique()[:10].tolist()
    print(f"{col:45s} | dtype: {str(cc_calls_df[col].dtype):8s} | nunique: {n_unique:6d} | nulls: {cc_calls_df[col].isnull().sum():6d} | samples: {sample_vals}")

   
## Data Cleaning and Type Enforcement

In [0]:
# ── Group 1: Fix numeric types and parse dates ──
cc_calls_df['Contact_ID'] = cc_calls_df['Contact_ID'].astype('int64')
cc_calls_df['Call_Date'] = pd.to_datetime(cc_calls_df['Call_Date'], format='%d-%m-%Y', errors='coerce')


In [0]:
# ── Group 2: Replace [Yes/No] → None, then fillna('Unknown') ──
yes_no_cleanup_cols = [
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_questionnaire_completion',
    'cc_dissatisfaction_time_to_complete',
]

for col in yes_no_cleanup_cols:
    cc_calls_df[col] = (
        cc_calls_df[col]
        .replace({'[Yes/No]': None})
        .fillna('Unknown')
    )

for col in yes_no_cleanup_cols:
    print(f"  {col}: {cc_calls_df[col].value_counts().to_dict()}")

In [0]:
# ── Group 3: Keyword yes/no matching for complex text columns ──
def clean_yes_no_keywords(series):
    """Lowercase, strip, then: contains 'yes' → Yes, contains 'no' → No, else Unknown."""
    cleaned = series.astype(str).str.lower().str.strip()
    return cleaned.apply(
        lambda x: 'Yes' if 'yes' in x
        else ('No' if 'no' in x or 'not applicable' in x or 'not mentioned' in x or 'not discussed' in x else 'Unknown')
    ).fillna('Unknown')

keyword_yes_no_cols = [
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_refund_discussed',
]

for col in keyword_yes_no_cols:
    cc_calls_df[col] = clean_yes_no_keywords(cc_calls_df[col])

print(f"Cleaned {len(keyword_yes_no_cols)} columns (keyword yes/no match) ✓")
for col in keyword_yes_no_cols:
    print(f"  {col}: {cc_calls_df[col].value_counts().to_dict()}")

In [0]:
# ── Group 4: Clean sentiment columns ──
# cc_contractor_sentiment: extract sentiment labels
cc_calls_df['cc_contractor_sentiment'] = (
    cc_calls_df['cc_contractor_sentiment']
    .astype(str)
    .str.lower()
    .str.strip()
)

def map_sentiment(x):
    if 'satisfied' in x:
        return 'Satisfied'
    elif 'dissatisfied' in x:
        return 'Dissatisfied'
    elif 'neutral' in x:
        return 'Neutral'
    elif 'not discussed' in x or 'not applicable' in x:
        return 'Not Discussed'
    else:
        return 'Unknown'

cc_calls_df['cc_contractor_sentiment'] = cc_calls_df['cc_contractor_sentiment'].apply(map_sentiment)

# Sentiment scores: convert to numeric or 'Unknown'
def clean_score(series):
    """Extract numeric scores, replace non-numeric with 'Unknown'."""
    return pd.to_numeric(series, errors='coerce').fillna(-1).astype(int).replace(-1, 'Unknown').astype(str)

score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score',
]

for col in score_cols:
    cc_calls_df[col] = clean_score(cc_calls_df[col])

# cc_pricing_mentioned and cc_pricing_sentiment_impact: extract Yes/No/Unknown
for col in ['cc_pricing_mentioned', 'cc_pricing_sentiment_impact']:
    cc_calls_df[col] = clean_yes_no_keywords(cc_calls_df[col])

print("Sentiment columns cleaned ✓")
print(f"  cc_contractor_sentiment: {cc_calls_df['cc_contractor_sentiment'].value_counts().to_dict()}")
print(f"  cc_pricing_mentioned: {cc_calls_df['cc_pricing_mentioned'].value_counts().to_dict()}")
print(f"  cc_pricing_sentiment_impact: {cc_calls_df['cc_pricing_sentiment_impact'].value_counts().to_dict()}")

In [0]:
# ── Group 5: Fill NaN → 'Unknown' for remaining columns ──
fillna_cols = [
    'cc_care_package',
    'cc_call_initiated_by',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained',
    'Co_Ref',
]

for col in fillna_cols:
    cc_calls_df[col] = cc_calls_df[col].fillna('Unknown')

print(f"Filled NaN → 'Unknown' for {len(fillna_cols)} columns ✓")

In [0]:
# ── Convert low-cardinality columns to 'category' dtype ──
CATEGORICAL_THRESHOLD = 30  # columns with <= 30 unique values

categorical_cols = []
for col in cc_calls_df.select_dtypes(include='object').columns:
    if cc_calls_df[col].nunique() <= CATEGORICAL_THRESHOLD:
        categorical_cols.append(col)

cc_calls_df[categorical_cols] = cc_calls_df[categorical_cols].astype('category')

print(f'Converted {len(categorical_cols)} columns to category:')
for c in categorical_cols:
    cats = cc_calls_df[c].cat.categories.tolist()
    print(f'  {c:50s} → {len(cats)} categories')

In [0]:
# ── Final dtype summary ──
print(f"Shape: {cc_calls_df.shape}\n")
cc_calls_df.info()

In [0]:
cc_calls_df.to_csv("../../data/processed/cc_calls.csv", index=False)